# Phase III — Empirical Implementation
## III.3 — Data Engineering

This notebook executes and validates the data-engineering layer of the extended
investment portfolio research project.

Research assumptions are governed by `docs/research_protocol.md`. This notebook
does not redefine methodological assumptions.

Current scope:

- **III.3.1 — Acquire Raw Data: COMPLETED**
  - Yahoo ETF acquisition/persistence: **COMPLETED**
  - FRED macroeconomic acquisition/persistence: **COMPLETED**
- **III.3.2 — Build Raw-Data Validation: IN PROGRESS**
  - Yahoo raw-data validation: **PASS**
  - FRED structural, persistence, coverage, and monthly-availability checks: **PASS**
  - FRED dtype / value-plausibility review: **REMAINING**
- **III.3.3 — Build the Analytical Dataset: NOT STARTED**

No monthly transformation, return calculation, portfolio construction,
macro-regime classification, rebalancing, turnover, or transaction-cost
calculation is performed in the raw-data work below.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: /Users/santiago/proyectos/multifactor-portfolio-research-v2


In [2]:
import pandas as pd

from src.config import (
    ANALYSIS_CONFIG,
    RAW_MACRO_DATA_FILE,
    RAW_MARKET_DATA_FILE,
)

from src.data.download_data import (
    download_market_data,
    get_market_data_start_date,
    get_market_data_end_date,
    save_raw_market_data,
)

from src.data.macro_data import (
    download_macro_data,
    get_fred_acquisition_window,
    save_raw_macro_data,
)

from src.data.load_data import (
    load_raw_macro_data,
    load_raw_market_data,
)

from src.data.validate_data import validate_raw_macro_data

from src.portfolio.portfolio_definitions import ELIGIBLE_ETFS

print("Data-engineering imports PASS")


Data-engineering imports PASS


## Data-Engineering Contract

The formal data-engineering specification and ownership contract was established
upstream and governs the implementation below.

The contract defines:

- canonical external data sources and identifiers;
- raw versus processed data boundaries;
- module ownership;
- analytical coverage rules;
- monthly observation policy;
- missing-data policy;
- validation responsibilities;
- blocker conditions;
- downstream exclusions.

Raw data preserves source evidence. Data-quality anomalies are detected and
investigated rather than silently repaired.


## III.3.1 — Acquire Raw Data

**Status: COMPLETED**

The formal acquisition stage requires both market and macroeconomic raw inputs.

Current state:

- Yahoo Finance ETF acquisition/persistence: **COMPLETED**
- FRED macroeconomic acquisition/persistence: **COMPLETED**

### Yahoo Finance Market Data

Yahoo Finance supplies daily market observations for the canonical ETF universe:

- SPY
- MTUM
- USMV
- QUAL
- AGG

Acquisition uses an input buffer beginning one calendar month before the
analytical sample so that the first potentially valid July 2013 monthly return
can later be constructed.

Yahoo's exclusive `end` convention is handled by the acquisition layer.
Source-level missing observations are preserved and automatic Yahoo repair is disabled.


In [3]:
print("Analysis start:", ANALYSIS_CONFIG.start_date)
print("Analysis end:", ANALYSIS_CONFIG.end_date)

print("Acquisition start:", get_market_data_start_date())
print("Exclusive acquisition end:", get_market_data_end_date())

print("Canonical ETF universe:", ELIGIBLE_ETFS)


Analysis start: 2013-07-01
Analysis end: 2024-12-31
Acquisition start: 2013-06-01
Exclusive acquisition end: 2025-01-01
Canonical ETF universe: ('SPY', 'MTUM', 'USMV', 'QUAL', 'AGG')


### Controlled Yahoo Acquisition


In [4]:
market_data = download_market_data()

print("Shape:", market_data.shape)
print("Start observation:", market_data.index.min())
print("End observation:", market_data.index.max())

print(
    "Returned tickers:",
    list(dict.fromkeys(market_data.columns.get_level_values(0)))
)

print(
    "Returned fields:",
    list(dict.fromkeys(market_data.columns.get_level_values(1)))
)


Shape: (2916, 40)
Start observation: 2013-06-03 00:00:00
End observation: 2024-12-31 00:00:00
Returned tickers: ['USMV', 'SPY', 'QUAL', 'MTUM', 'AGG']
Returned fields: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Capital Gains']


### Raw Market-Data Persistence

The acquired Yahoo dataset is persisted without analytical transformation.

The raw artifact must preserve:

- dates;
- ticker identity;
- field structure;
- source-level missing values;
- corporate-action fields;
- numerical observations.

The configured artifact is stored as Parquet under `RAW_DATA_DIR`.


In [5]:
save_raw_market_data(market_data)

print("Raw file:", RAW_MARKET_DATA_FILE)
print("Yahoo raw-data persistence completed")


Raw file: /Users/santiago/proyectos/multifactor-portfolio-research-v2/data/raw/yahoo_market_data.parquet
Yahoo raw-data persistence completed


## III.3.2 — Build Raw-Data Validation

**Status: IN PROGRESS**

Yahoo raw-data validation has passed.

FRED raw-data validation has passed the checks implemented in this notebook for:

- required series and non-empty structure;
- `DatetimeIndex` structure;
- chronological ordering;
- duplicate dates;
- persistence preservation;
- required calendar-month coverage;
- monthly availability for both required series;
- explicit reporting of source-level missing observations.

Formal III.3.2 remains open pending the final FRED dtype / value-plausibility
review and consolidated exit assessment.

Validation detects and reports source behavior without silently repairing the data.


### Yahoo Schema and Universe Validation


In [6]:
returned_tickers = set(
    market_data.columns.get_level_values(0)
)

expected_tickers = set(ELIGIBLE_ETFS)

assert returned_tickers == expected_tickers
assert isinstance(market_data.columns, pd.MultiIndex)
assert market_data.columns.nlevels == 2

print("Canonical ETF membership PASS")
print("MultiIndex schema PASS")
print("Observed width:", market_data.shape[1])


Canonical ETF membership PASS
MultiIndex schema PASS
Observed width: 40


### Early-History and Missingness Inspection

Missing or unavailable source observations are treated as data-quality events,
not automatically repaired.

QUAL begins later than the other ETFs in the acquisition window. Its unavailable
pre-inception observations are intentionally preserved.

No forward filling, interpolation, synthetic reconstruction, zero-return
replacement, or weight renormalization is performed here.


In [7]:
close_missing_counts = {
    ticker: int(market_data[ticker]["Close"].isna().sum())
    for ticker in ELIGIBLE_ETFS
}

first_valid_close = {
    ticker: market_data[ticker]["Close"].first_valid_index()
    for ticker in ELIGIBLE_ETFS
}

print("Missing Close observations:")
for ticker, count in close_missing_counts.items():
    print(f"{ticker}: {count}")

print("\nFirst valid Close observation:")
for ticker, first_date in first_valid_close.items():
    print(f"{ticker}: {first_date}")


Missing Close observations:
SPY: 0
MTUM: 0
USMV: 0
QUAL: 32
AGG: 0

First valid Close observation:
SPY: 2013-06-03 00:00:00
MTUM: 2013-06-03 00:00:00
USMV: 2013-06-03 00:00:00
QUAL: 2013-07-18 00:00:00
AGG: 2013-06-03 00:00:00


### Raw Persistence Reload


In [8]:
reloaded_market_data = load_raw_market_data()

print("Original shape:", market_data.shape)
print("Reloaded shape:", reloaded_market_data.shape)

assert market_data.shape == reloaded_market_data.shape

print("Reload shape preservation PASS")


Original shape: (2916, 40)
Reloaded shape: (2916, 40)
Reload shape preservation PASS


### Parquet Timestamp-Resolution Observation

During validation, the Parquet round trip changed the pandas `DatetimeIndex`
resolution representation from `datetime64[s]` to `datetime64[ms]`.

Investigation confirmed that the timestamp values themselves were unchanged.

This is classified as a storage-representation difference rather than empirical
data loss.

For preservation validation only, both indices are converted to a common
`datetime64[ns]` representation before strict DataFrame equality is tested.
No source observation is altered in the persisted raw dataset.


In [9]:
print("Original index dtype:", market_data.index.dtype)
print("Reloaded index dtype:", reloaded_market_data.index.dtype)

print("Original first date:", market_data.index[0])
print("Reloaded first date:", reloaded_market_data.index[0])

print("Original last date:", market_data.index[-1])
print("Reloaded last date:", reloaded_market_data.index[-1])

print(
    "Same index length:",
    len(market_data.index) == len(reloaded_market_data.index),
)


Original index dtype: datetime64[s]
Reloaded index dtype: datetime64[ms]
Original first date: 2013-06-03 00:00:00
Reloaded first date: 2013-06-03 00:00:00
Original last date: 2024-12-31 00:00:00
Reloaded last date: 2024-12-31 00:00:00
Same index length: True


### Timestamp-Value and Full-Frame Preservation


In [10]:
original_for_validation = market_data.copy()
reloaded_for_validation = reloaded_market_data.copy()

original_for_validation.index = (
    original_for_validation.index.astype("datetime64[ns]")
)

reloaded_for_validation.index = (
    reloaded_for_validation.index.astype("datetime64[ns]")
)

pd.testing.assert_index_equal(
    original_for_validation.index,
    reloaded_for_validation.index,
)

pd.testing.assert_frame_equal(
    original_for_validation,
    reloaded_for_validation,
)

print("Index timestamp values PASS")
print("Raw market-data persistence PASS")


Index timestamp values PASS
Raw market-data persistence PASS


### Targeted Preservation Checks

In addition to full-frame equality, two research-specific properties are checked
explicitly:

1. QUAL's early missing observations survive persistence unchanged.
2. Yahoo corporate-action fields remain present after reload.


In [11]:
original_qual_na = market_data["QUAL"]["Close"].isna().sum()
reloaded_qual_na = reloaded_market_data["QUAL"]["Close"].isna().sum()

assert original_qual_na == reloaded_qual_na

print(
    "QUAL missingness preservation PASS:",
    original_qual_na,
)


QUAL missingness preservation PASS: 32


In [12]:
required_action_fields = {
    "Dividends",
    "Stock Splits",
    "Capital Gains",
}

original_fields = set(
    market_data.columns.get_level_values(1)
)

reloaded_fields = set(
    reloaded_market_data.columns.get_level_values(1)
)

assert required_action_fields.issubset(original_fields)
assert required_action_fields.issubset(reloaded_fields)

print("Corporate-action fields preservation PASS")


Corporate-action fields preservation PASS


### Yahoo Raw-Data Checkpoint


In [13]:
print("Yahoo Raw Acquisition & Validation")
print("----------------------------------")
print("Yahoo acquisition                 PASS")
print("Canonical ETF universe            PASS")
print("Raw persistence                   PASS")
print("Timestamp values preserved        PASS")
print("QUAL missingness preserved        PASS")
print("Corporate-action fields           PASS")
print("Prohibited imputation             NONE")
print("Automatic Yahoo repair            DISABLED")
print()
print("Yahoo raw-data checkpoint: COMPLETED")


Yahoo Raw Acquisition & Validation
----------------------------------
Yahoo acquisition                 PASS
Canonical ETF universe            PASS
Raw persistence                   PASS
Timestamp values preserved        PASS
QUAL missingness preserved        PASS
Corporate-action fields           PASS
Prohibited imputation             NONE
Automatic Yahoo repair            DISABLED

Yahoo raw-data checkpoint: COMPLETED


## FRED Macroeconomic Data — Raw Acquisition & Validation

The remaining III.3.1 raw source consists of the required FRED series:

- `DGS3MO`
- `T10Y3M`

The FRED acquisition window is derived from the frozen analytical period.
One pre-sample calendar month is included because the prior month-end `T10Y3M`
state will later classify the first analytical return month. This buffer does
not extend the analytical sample.

At this stage the notebook only acquires, persists, reloads, and validates raw
source observations. It does **not** perform monthly observation selection,
risk-free conversion, regime classification, imputation, portfolio alignment,
or analytical transformation.


In [14]:
fred_start, fred_end = get_fred_acquisition_window()

print("FRED acquisition window:")
print(f"  Start: {fred_start}")
print(f"  End:   {fred_end}")

macro_data = download_macro_data()

print("Shape:", macro_data.shape)
print("Columns:", list(macro_data.columns))
display(macro_data.head())
display(macro_data.tail())

save_raw_macro_data(macro_data)

print("Raw file:", RAW_MACRO_DATA_FILE)
print("FRED raw-data persistence completed")


FRED acquisition window:
  Start: 2013-06-01
  End:   2024-12-31
Shape: (3022, 2)
Columns: ['DGS3MO', 'T10Y3M']


,DGS3MO,T10Y3M
DATE,,
2013-06-03,0.05,2.08
2013-06-04,0.04,2.10
2013-06-05,0.05,2.05
2013-06-06,0.05,2.03
2013-06-07,0.04,2.13


,DGS3MO,T10Y3M
DATE,,
2024-12-25,NaN,NaN
2024-12-26,4.35,0.23
2024-12-27,4.31,0.31
2024-12-30,4.37,0.18
2024-12-31,4.37,0.21


Raw file: /Users/santiago/proyectos/multifactor-portfolio-research-v2/data/raw/fred_macro_data.parquet
FRED raw-data persistence completed


In [15]:
reloaded_macro_data = load_raw_macro_data()

print("Original shape:", macro_data.shape)
print("Reloaded shape:", reloaded_macro_data.shape)

assert macro_data.shape == reloaded_macro_data.shape

print("Raw FRED reload PASS")


Original shape: (3022, 2)
Reloaded shape: (3022, 2)
Raw FRED reload PASS


In [16]:
pd.testing.assert_frame_equal(
    macro_data,
    reloaded_macro_data,
)

print("Raw FRED persistence preservation PASS")


Raw FRED persistence preservation PASS


In [17]:
validate_raw_macro_data(reloaded_macro_data)


Raw FRED structural validation PASS

Coverage:
  Required start month: 2013-06
  Actual start month:   2013-06
  Required end month:   2024-12
  Actual end month:     2024-12

Missing observations:
DGS3MO    125
T10Y3M    125
dtype: int64


In [18]:
monthly_valid_counts = (
    reloaded_macro_data
    .notna()
    .groupby(reloaded_macro_data.index.to_period("M"))
    .sum()
)

months_without_dgs3mo = monthly_valid_counts.index[
    monthly_valid_counts["DGS3MO"] == 0
]

months_without_t10y3m = monthly_valid_counts.index[
    monthly_valid_counts["T10Y3M"] == 0
]

assert len(months_without_dgs3mo) == 0
assert len(months_without_t10y3m) == 0

print("Monthly FRED availability inspection")
print("------------------------------------")
print(f"Months inspected:                 {len(monthly_valid_counts)}")
print(f"Months without valid DGS3MO:      {len(months_without_dgs3mo)}")
print(f"Months without valid T10Y3M:      {len(months_without_t10y3m)}")
print()
print("Required monthly FRED availability PASS")


Monthly FRED availability inspection
------------------------------------
Months inspected:                 139
Months without valid DGS3MO:      0
Months without valid T10Y3M:      0

Required monthly FRED availability PASS


In [19]:
import numpy as np

print("FRED dtype & value-plausibility inspection")
print("------------------------------------------")

# 1. Data types
print("\nData types:")
print(reloaded_macro_data.dtypes)

# 2. Numeric representation
non_numeric_columns = [
    column
    for column in reloaded_macro_data.columns
    if not pd.api.types.is_numeric_dtype(reloaded_macro_data[column])
]

print("\nNon-numeric columns:")
print(non_numeric_columns)

# 3. Infinite values
infinite_counts = pd.Series(
    {
        column: np.isinf(reloaded_macro_data[column].dropna()).sum()
        for column in reloaded_macro_data.columns
    }
)

print("\nInfinite values:")
print(infinite_counts)

# 4. Descriptive ranges
print("\nValue ranges:")
display(
    reloaded_macro_data.agg(
        ["min", "max", "mean", "median"]
    )
)

# 5. Automated technical assertions
assert len(non_numeric_columns) == 0
assert infinite_counts.sum() == 0

print("\nFRED dtype / finite-value validation PASS")

FRED dtype & value-plausibility inspection
------------------------------------------

Data types:
DGS3MO    float64
T10Y3M    float64
dtype: object

Non-numeric columns:
[]

Infinite values:
DGS3MO    0
T10Y3M    0
dtype: int64

Value ranges:


,DGS3MO,T10Y3M
min,0.000000,-1.890000
max,5.630000,2.970000
mean,1.587377,0.906831
median,0.540000,1.170000



FRED dtype / finite-value validation PASS


## Raw-Data Checkpoint — Final Result

### III.3.1 — Acquire Raw Data

**CLOSED — PASS**

Both required raw-data sources now have reproducible acquisition and persistence paths:

- Yahoo Finance market data;
- FRED `DGS3MO` and `T10Y3M`.

Yahoo acquisition preserves the canonical ETF universe, source-level missingness,
corporate-action fields, and the required pre-sample price buffer.

FRED acquisition derives its required pre-sample month from the frozen analytical
start date rather than introducing an independent hard-coded research parameter.

---

### III.3.2 — Build Raw-Data Validation

**CLOSED — PASS**

Yahoo raw-data validation passed the required acquisition, persistence, timestamp,
missingness, and corporate-action preservation checks.

FRED raw-data validation establishes:

- required `DGS3MO` and `T10Y3M` series present;
- acquisition window correctly derived as June 2013 through December 2024;
- Parquet persistence and reload operational;
- full-frame persistence preservation;
- valid `DatetimeIndex`;
- chronological ordering;
- no duplicate dates;
- required calendar-month coverage from June 2013 through December 2024;
- 139 required calendar months inspected;
- zero months without a valid `DGS3MO` observation;
- zero months without a valid `T10Y3M` observation;
- source-level daily missing observations explicitly reported and preserved;
- numeric `float64` representation for both FRED series;
- no non-numeric required series;
- no infinite values;
- observed value ranges reviewed as economically plausible for the intended series.

Observed FRED daily missingness:

- `DGS3MO`: 125 observations;
- `T10Y3M`: 125 observations.

These missing daily observations do not create missing required calendar months
and therefore do not prevent application of the frozen monthly
last-available-observation methodology.

No imputation, forward fill, interpolation, synthetic reconstruction, or silent
data repair has been introduced.

The FRED dtype and finite-value checks passed programmatically. Observed empirical
ranges were reviewed for economic plausibility without introducing arbitrary
hard-coded plausibility thresholds.

---

### III.3 Raw-Data Gate

**PASS**

III.3.1 and III.3.2 are closed.

The project may now proceed to:

> **III.3.3 — Build the Analytical Dataset**

The next stage will implement the frozen transformation rules for monthly market
and macroeconomic data, including monthly observation selection, ETF return
calculation, DGS3MO monthly risk-free construction, lagged T10Y3M regime
assignment, and source alignment.

Portfolio construction, rebalancing, turnover, transaction costs, and performance
analysis remain downstream and must not begin during III.3.3.